## Import Libraries


In [1]:
import os
import cv2
import time
import numpy as np
import urllib.request

## Load Haar Cascade XML File for Frontal Faces


In [2]:
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
print(f"Haar Cascade XML file downloaded at: {cascade_path}")

haar_cascade_model = cv2.CascadeClassifier(cascade_path)
if haar_cascade_model.empty():
    print("Error: Failed to load Haar Cascade!!!")
    exit()

print("Successfully loaded Haar Cascade")

Haar Cascade XML file downloaded at: /home/debargha/.edu/lib/python3.12/site-packages/cv2/data/haarcascade_frontalface_default.xml
Successfully loaded Haar Cascade


## Load Deep Neural Network Classifier


In [3]:
prototxt = "deploy.prototxt"
model = "res10_300x300_ssd_iter_140000.caffemodel"

if not os.path.exists(prototxt):
    print(f"Loading {prototxt}...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
        prototxt,
    )
    print(f"Successfully loaded '{prototxt}'")
elif os.path.exists(prototxt):
    print(f"'{prototxt}' already exists")
else:
    print(f"Failed to load '{prototxt}'!!!")

if not os.path.exists(model):
    print(f"Loading {model}...")
    urllib.request.urlretrieve(
        "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
        model,
    )
    print(f"Successfully loaded '{model}'")
elif os.path.exists(model):
    print(f"'{model}' already exists")
else:
    print(f"Failed to load '{model}'!!!")

Loading deploy.prototxt...
Successfully loaded 'deploy.prototxt'
Loading res10_300x300_ssd_iter_140000.caffemodel...
Successfully loaded 'res10_300x300_ssd_iter_140000.caffemodel'


## Load the DNN Model


In [4]:
dnn_net = cv2.dnn.readNetFromCaffe(prototxt, caffeModel=model)

## FPS Comparison of Haar Cascade vs DNN Classifier


In [ ]:
# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Resize for faster processing
    frame = cv2.resize(frame, (640, 480))
    (h, w) = frame.shape[:2]

    # Haar Cascade Classifier
    start_haar = time.time()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = haar_cascade_model.detectMultiScale(gray, 1.3, 5)

    for x, y, w1, h1 in faces:
        cv2.rectangle(frame, (x, y), (x + w1, y + h1), (0, 255, 0), 2)

    fps_haar = 1 / (time.time() - start_haar)

    # DNN Classifier
    start_dnn = time.time()

    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))

    dnn_net.setInput(blob)
    detections = dnn_net.forward()

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > 0.5:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (x1, y1, x2, y2) = box.astype("int")

            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

    fps_dnn = 1 / (time.time() - start_dnn)

    # Display FPS
    cv2.putText(
        frame,
        f"Haar Cascade FPS: {fps_haar:.2f}",
        (10, 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2,
    )

    cv2.putText(
        frame,
        f"DNN FPS: {fps_dnn:.2f}",
        (10, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 0, 0),
        2,
    )

    cv2.imshow("Human Face Detection using Haar Cascade vs DNN Classifier", frame)

    # Exit using ESC key
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

## Side-by-Side FPS Comparison of Haar Cascade vs DNN Classifier


In [6]:
# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (640, 480))
    (h, w) = frame.shape[:2]

    # Create copies for each method
    haar_frame = frame.copy()
    dnn_frame = frame.copy()

    # Haar Cascade Classifier
    start_haar = time.time()

    gray = cv2.cvtColor(haar_frame, cv2.COLOR_BGR2GRAY)
    faces = haar_cascade_model.detectMultiScale(gray, 1.3, 5)

    for x, y, w1, h1 in faces:
        cv2.rectangle(haar_frame, (x, y), (x + w1, y + h1), (0, 255, 0), 2)

    fps_haar = 1 / (time.time() - start_haar)

    cv2.putText(
        haar_frame,
        f"Haar FPS: {fps_haar:.2f}",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2,
    )

    cv2.putText(
        haar_frame,
        "Haar Cascade",
        (10, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2,
    )

    # DNN Classifer
    start_dnn = time.time()

    blob = cv2.dnn.blobFromImage(dnn_frame, 1.0, (300, 300), (104.0, 177.0, 123.0))

    dnn_net.setInput(blob)
    detections = dnn_net.forward()

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > 0.5:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (x1, y1, x2, y2) = box.astype("int")

            cv2.rectangle(dnn_frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

    fps_dnn = 1 / (time.time() - start_dnn)

    cv2.putText(
        dnn_frame,
        f"DNN FPS: {fps_dnn:.2f}",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2,
    )

    cv2.putText(
        dnn_frame,
        "DNN (Deep Learning)",
        (10, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2,
    )

    # Combine side by side
    combined = np.hstack((haar_frame, dnn_frame))

    cv2.imshow(
        "Side-by-Side Comparision of Human Face Detection using Haar Cascade vs DNN Classifier",
        combined,
    )

    # Exit using ESC key
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()